# Chunking Strategy Comparison: Chunk Size & Overlap

## Purpose

This notebook evaluates how chunk size and overlap parameters affect search result quality. The current system uses `chunk_size=500` and `chunk_overlap=50` with `RecursiveCharacterTextSplitter`. This experiment tests whether different configurations produce measurably different search results.

The experiment runs in two phases:

| Phase | What It Tests | Configurations |
|-------|--------------|----------------|
| **Phase 1: Chunk Size** | Three chunk sizes with proportional ~10% overlap | 256/25, 512/50, 1024/100 |
| **Phase 2: Overlap Sensitivity** | Four overlap values for the best chunk size from Phase 1 | 0%, ~10%, ~20%, ~40% of chunk size |

## Prerequisites

- FastAPI server running on `localhost:8000`
- FAISS+BM25 search index populated with sagemaker-docs
- `pip install -r requirements.txt`

## Important Notes

- **Runtime**: ~20-30 minutes total (each configuration requires a full delete-reindex-search cycle)
- **Destructive**: This notebook clears and re-indexes the search index multiple times
- **Restoration**: The final cell restores the default configuration (chunk_size=500, chunk_overlap=50)

In [ ]:
import sys
import time

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

# Add experiments/ to path so helpers can be imported
sys.path.insert(0, ".")
from helpers import (
    search,
    results_to_dataframe,
    hits_to_doc_ids,
    jaccard_similarity,
    overlap_matrix,
    rank_biased_overlap,
    clear_index,
    reindex_local_docs,
    get_index_stats,
    run_query_suite,
    BASE_URL,
)

# Verify the API is reachable
stats = get_index_stats()
print(f"Index: {stats['index_name']}")
print(f"Documents: {stats['doc_count']}")
print(f"Status: {stats['status']}")

## Configuration Design

### Phase 1: Chunk Size Comparison

We test three chunk sizes with proportional overlap (~10% of chunk size). The rationale:

| Config | chunk_size | chunk_overlap | Rationale |
|--------|-----------|---------------|----------|
| **Small (256/25)** | 256 | 25 | Fine-grained chunks — more chunks per document, higher precision, but each chunk carries less context |
| **Medium (512/50)** | 512 | 50 | Close to current production (500/50) — balanced tradeoff |
| **Large (1024/100)** | 1024 | 100 | Coarser chunks — fewer chunks per document, more context per chunk, but may dilute relevance |

### Phase 2: Overlap Sensitivity

For the best chunk_size from Phase 1 (defaults to 512), we test four overlap values:

| chunk_overlap | % of chunk_size | Expected Effect |
|--------------|----------------|------------------|
| 0 | 0% | No redundancy — risk of losing context at chunk boundaries |
| 50 | ~10% | Minimal overlap (current default) |
| 100 | ~20% | Moderate overlap — more boundary context preserved |
| 200 | ~40% | High overlap — significant chunk redundancy, larger index |

In [ ]:
# Phase 1: Chunk size comparison (proportional overlap at ~10%)
CHUNK_SIZE_CONFIGS = [
    {"label": "small_256_25",   "chunk_size": 256,  "chunk_overlap": 25},
    {"label": "medium_512_50",  "chunk_size": 512,  "chunk_overlap": 50},
    {"label": "large_1024_100", "chunk_size": 1024, "chunk_overlap": 100},
]

# Same 9 queries from search_type_comparison.ipynb
QUERIES = {
    "text_excelling": [
        "RetainAllVariantProperties",
        "SAGEMAKER_NOTEBOOK_NO_DIRECT_INTERNET_ACCESS",
        "AmazonSageMakerFullAccess",
    ],
    "vector_excelling": [
        "How do I make sure my notebook isn't exposed to the internet?",
        "What is the benefit of using a project instead of running pipelines directly?",
        "How can data scientists share code consistently across a team?",
    ],
    "hybrid_excelling": [
        "What IAM permissions does an execution role need to run a training job?",
        "How does EventBridge trigger actions when an endpoint changes status?",
        "What kubectl commands do I use to check a training job running in Kubernetes?",
    ],
}

ALL_QUERIES = [q for qs in QUERIES.values() for q in qs]
ALL_CATEGORIES = [cat for cat, qs in QUERIES.items() for _ in qs]
SEARCH_TYPE = "hybrid"
RESULT_SIZE = 10

print(f"Total queries: {len(ALL_QUERIES)}")
print(f"Search type: {SEARCH_TYPE}")
print(f"Results per query: {RESULT_SIZE}")
print(f"Phase 1 configs: {len(CHUNK_SIZE_CONFIGS)}")

---
## Phase 1: Chunk Size Comparison

For each configuration, we:
1. Clear all documents from the index
2. Re-index all sagemaker-docs with the new parameters
3. Wait for the search index to settle
4. Run all 9 test queries
5. Store results for comparison

In [ ]:
phase1_results = {}   # label -> {query -> search_response}
phase1_stats = {}     # label -> {chunk_size, chunk_overlap, total_chunks, ...}
phase1_timings = {}   # label -> seconds

for config in CHUNK_SIZE_CONFIGS:
    label = config["label"]
    print(f"\n{'='*60}")
    print(f"Configuration: {label} (chunk_size={config['chunk_size']}, overlap={config['chunk_overlap']})")
    print(f"{'='*60}")

    start = time.time()

    # Step 1: Clear index
    print("  Clearing index...")
    clear_result = clear_index()
    print(f"  Deleted {clear_result['total_chunks_deleted']} chunks from {clear_result['filenames_deleted']} files")

    # Step 2: Re-index with new params
    print(f"  Re-indexing with chunk_size={config['chunk_size']}, overlap={config['chunk_overlap']}...")
    index_result = reindex_local_docs(
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
    )

    # Step 3: Let search index settle
    time.sleep(5)

    stats = get_index_stats()
    phase1_stats[label] = {
        "chunk_size": config["chunk_size"],
        "chunk_overlap": config["chunk_overlap"],
        "total_chunks": stats["doc_count"],
        "indexed_count": index_result["indexed_count"],
        "skipped_count": index_result["skipped_count"],
    }
    print(f"  Indexed: {stats['doc_count']} total chunks")

    # Step 4: Run queries
    print("  Running queries...")
    phase1_results[label] = run_query_suite(ALL_QUERIES, search_type=SEARCH_TYPE, size=RESULT_SIZE)

    elapsed = time.time() - start
    phase1_timings[label] = elapsed
    print(f"  Done in {elapsed:.1f}s")

print(f"\nTotal Phase 1 time: {sum(phase1_timings.values()):.1f}s")

### Chunk Statistics

How does chunk size affect the total number of indexed chunks? Smaller chunks produce more chunks per document (higher granularity but more index entries).

In [ ]:
stats_df = pd.DataFrame([
    {"Config": label, **phase1_stats[label]}
    for label in phase1_stats
])
display(stats_df)

# Bar chart: total chunks per config
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#4C72B0", "#55A868", "#C44E52"]
ax.bar(stats_df["Config"], stats_df["total_chunks"], color=colors)
ax.set_ylabel("Total Chunks")
ax.set_title("Total Indexed Chunks by Configuration")
for i, v in enumerate(stats_df["total_chunks"]):
    ax.text(i, v + 5, str(v), ha="center", fontsize=11)
plt.tight_layout()
plt.show()

### Side-by-Side Results

For each query, compare the top-5 results across chunk size configurations. Look for:
- Do different chunk sizes retrieve different documents?
- Do larger chunks improve context for semantic queries?
- Do smaller chunks improve precision for exact-match queries?

In [ ]:
config_labels = [c["label"] for c in CHUNK_SIZE_CONFIGS]

for query in ALL_QUERIES:
    print(f"\n{'='*90}")
    print(f"Query: {query}")
    print(f"{'='*90}")
    for label in config_labels:
        resp = phase1_results[label][query]
        df = results_to_dataframe(resp)
        print(f"\n--- {label} ({resp['total_hits']} total hits) ---")
        display(df.head(5))

### Overlap Analysis Between Configurations

How much do result sets overlap between chunk size configurations? High overlap means chunk size has little impact on which documents are retrieved; low overlap means the configurations retrieve meaningfully different content.

- **Jaccard Similarity**: Set-based overlap (ignores ranking)
- **Rank-Biased Overlap (RBO)**: Position-weighted overlap (top ranks matter more)

In [ ]:
# Per-query Jaccard matrices between configs
jaccard_matrices = []
rbo_data = []

for query in ALL_QUERIES:
    ids = {
        label: hits_to_doc_ids(phase1_results[label][query])
        for label in config_labels
    }
    jm = overlap_matrix(ids)
    jaccard_matrices.append(jm)

    # Pairwise RBO
    for i, a in enumerate(config_labels):
        for b in config_labels[i + 1:]:
            rbo_val = rank_biased_overlap(ids[a], ids[b])
            rbo_data.append({
                "query": query[:50],
                "pair": f"{a} vs {b}",
                "rbo": round(rbo_val, 3),
            })

# Average Jaccard matrix across all queries
avg_jaccard = sum(jaccard_matrices) / len(jaccard_matrices)
print("Average Jaccard Similarity Matrix (across all 9 queries):")
display(avg_jaccard.round(3))

In [ ]:
# Jaccard heatmap
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(avg_jaccard.values, cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(len(config_labels)))
ax.set_yticks(range(len(config_labels)))
ax.set_xticklabels(config_labels, rotation=30, ha="right")
ax.set_yticklabels(config_labels)
for i in range(len(config_labels)):
    for j in range(len(config_labels)):
        ax.text(j, i, f"{avg_jaccard.values[i, j]:.3f}", ha="center", va="center", fontsize=12)
plt.colorbar(im, ax=ax, label="Jaccard Similarity")
ax.set_title("Result Set Overlap Between Chunk Size Configs")
plt.tight_layout()
plt.show()

In [ ]:
# RBO comparison table
rbo_df = pd.DataFrame(rbo_data)
print("Rank-Biased Overlap (RBO, p=0.9) per query:")
display(rbo_df.pivot(index="query", columns="pair", values="rbo").round(3))

### Score Distribution Analysis

How do relevance scores compare across chunk size configurations and query categories?

In [ ]:
# Collect scores by config and query category
score_data = []
for label in config_labels:
    for cat, query in zip(ALL_CATEGORIES, ALL_QUERIES):
        resp = phase1_results[label][query]
        for hit in resp.get("hits", []):
            score_data.append({
                "config": label,
                "category": cat,
                "score": hit["score"],
            })

score_df = pd.DataFrame(score_data)

# Box plot: score distribution by config, one subplot per category
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
categories = list(QUERIES.keys())
category_labels = ["Text-Excelling", "Vector-Excelling", "Hybrid-Excelling"]

for ax, cat, cat_label in zip(axes, categories, category_labels):
    cat_df = score_df[score_df["category"] == cat]
    cat_df.boxplot(column="score", by="config", ax=ax)
    ax.set_title(cat_label)
    ax.set_xlabel("Configuration")
    ax.set_ylabel("Score")
    ax.tick_params(axis="x", rotation=30)

plt.suptitle("Score Distributions by Query Category and Chunk Size", fontsize=14)
plt.tight_layout()
plt.show()

---
## Phase 2: Overlap Sensitivity

Based on Phase 1 analysis, we pick the best chunk_size (default: 512) and test how sensitive results are to overlap size. Higher overlap preserves more context at chunk boundaries but increases index size.

Update `BEST_CHUNK_SIZE` below after analyzing Phase 1 results.

In [ ]:
# Update this after analyzing Phase 1 results
BEST_CHUNK_SIZE = 512

OVERLAP_CONFIGS = [
    {"label": f"cs{BEST_CHUNK_SIZE}_ov0",   "chunk_size": BEST_CHUNK_SIZE, "chunk_overlap": 0},
    {"label": f"cs{BEST_CHUNK_SIZE}_ov50",  "chunk_size": BEST_CHUNK_SIZE, "chunk_overlap": 50},
    {"label": f"cs{BEST_CHUNK_SIZE}_ov100", "chunk_size": BEST_CHUNK_SIZE, "chunk_overlap": 100},
    {"label": f"cs{BEST_CHUNK_SIZE}_ov200", "chunk_size": BEST_CHUNK_SIZE, "chunk_overlap": 200},
]

print(f"Best chunk_size from Phase 1: {BEST_CHUNK_SIZE}")
print(f"Phase 2 configs: {len(OVERLAP_CONFIGS)}")
for c in OVERLAP_CONFIGS:
    pct = round(c['chunk_overlap'] / c['chunk_size'] * 100)
    print(f"  {c['label']}: overlap={c['chunk_overlap']} ({pct}% of chunk_size)")

In [ ]:
phase2_results = {}
phase2_stats = {}
phase2_timings = {}

for config in OVERLAP_CONFIGS:
    label = config["label"]
    print(f"\n{'='*60}")
    print(f"Configuration: {label} (chunk_size={config['chunk_size']}, overlap={config['chunk_overlap']})")
    print(f"{'='*60}")

    start = time.time()

    print("  Clearing index...")
    clear_result = clear_index()
    print(f"  Deleted {clear_result['total_chunks_deleted']} chunks from {clear_result['filenames_deleted']} files")

    print(f"  Re-indexing with chunk_size={config['chunk_size']}, overlap={config['chunk_overlap']}...")
    index_result = reindex_local_docs(
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
    )

    # Wait for search index to settle
    time.sleep(5)

    stats = get_index_stats()
    phase2_stats[label] = {
        "chunk_size": config["chunk_size"],
        "chunk_overlap": config["chunk_overlap"],
        "total_chunks": stats["doc_count"],
        "indexed_count": index_result["indexed_count"],
        "skipped_count": index_result["skipped_count"],
    }
    print(f"  Indexed: {stats['doc_count']} total chunks")

    print("  Running queries...")
    phase2_results[label] = run_query_suite(ALL_QUERIES, search_type=SEARCH_TYPE, size=RESULT_SIZE)

    elapsed = time.time() - start
    phase2_timings[label] = elapsed
    print(f"  Done in {elapsed:.1f}s")

print(f"\nTotal Phase 2 time: {sum(phase2_timings.values()):.1f}s")

### Phase 2: Chunk Statistics

In [ ]:
overlap_stats_df = pd.DataFrame([
    {"Config": label, **phase2_stats[label]}
    for label in phase2_stats
])
display(overlap_stats_df)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]
ax.bar(overlap_stats_df["Config"], overlap_stats_df["total_chunks"], color=colors)
ax.set_ylabel("Total Chunks")
ax.set_title(f"Total Indexed Chunks by Overlap (chunk_size={BEST_CHUNK_SIZE})")
for i, v in enumerate(overlap_stats_df["total_chunks"]):
    ax.text(i, v + 5, str(v), ha="center", fontsize=11)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### Phase 2: Side-by-Side Results

In [ ]:
overlap_labels = [c["label"] for c in OVERLAP_CONFIGS]

for query in ALL_QUERIES:
    print(f"\n{'='*90}")
    print(f"Query: {query}")
    print(f"{'='*90}")
    for label in overlap_labels:
        resp = phase2_results[label][query]
        df = results_to_dataframe(resp)
        print(f"\n--- {label} ({resp['total_hits']} total hits) ---")
        display(df.head(5))

### Phase 2: Overlap Analysis Between Overlap Configs

In [ ]:
jaccard_matrices_p2 = []
rbo_data_p2 = []

for query in ALL_QUERIES:
    ids = {
        label: hits_to_doc_ids(phase2_results[label][query])
        for label in overlap_labels
    }
    jm = overlap_matrix(ids)
    jaccard_matrices_p2.append(jm)

    for i, a in enumerate(overlap_labels):
        for b in overlap_labels[i + 1:]:
            rbo_val = rank_biased_overlap(ids[a], ids[b])
            rbo_data_p2.append({
                "query": query[:50],
                "pair": f"{a} vs {b}",
                "rbo": round(rbo_val, 3),
            })

avg_jaccard_p2 = sum(jaccard_matrices_p2) / len(jaccard_matrices_p2)
print("Average Jaccard Similarity Matrix (across all 9 queries):")
display(avg_jaccard_p2.round(3))

# Heatmap
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(avg_jaccard_p2.values, cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(len(overlap_labels)))
ax.set_yticks(range(len(overlap_labels)))
ax.set_xticklabels(overlap_labels, rotation=30, ha="right")
ax.set_yticklabels(overlap_labels)
for i in range(len(overlap_labels)):
    for j in range(len(overlap_labels)):
        ax.text(j, i, f"{avg_jaccard_p2.values[i, j]:.3f}", ha="center", va="center", fontsize=11)
plt.colorbar(im, ax=ax, label="Jaccard Similarity")
ax.set_title(f"Result Set Overlap Between Overlap Configs (chunk_size={BEST_CHUNK_SIZE})")
plt.tight_layout()
plt.show()

### Phase 2: Score Distribution

In [ ]:
score_data_p2 = []
for label in overlap_labels:
    for cat, query in zip(ALL_CATEGORIES, ALL_QUERIES):
        resp = phase2_results[label][query]
        for hit in resp.get("hits", []):
            score_data_p2.append({
                "config": label,
                "category": cat,
                "score": hit["score"],
            })

score_df_p2 = pd.DataFrame(score_data_p2)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
for ax, cat, cat_label in zip(axes, categories, category_labels):
    cat_df = score_df_p2[score_df_p2["category"] == cat]
    cat_df.boxplot(column="score", by="config", ax=ax)
    ax.set_title(cat_label)
    ax.set_xlabel("Configuration")
    ax.set_ylabel("Score")
    ax.tick_params(axis="x", rotation=30)

plt.suptitle(f"Score Distributions by Overlap (chunk_size={BEST_CHUNK_SIZE})", fontsize=14)
plt.tight_layout()
plt.show()

---
## Restore Default Index

Re-index with the current production defaults so the system is left in a usable state.

In [ ]:
print("Restoring default index configuration (chunk_size=500, chunk_overlap=50)...")
clear_index()
reindex_local_docs(chunk_size=500, chunk_overlap=50)
stats = get_index_stats()
print(f"Restored: {stats['doc_count']} chunks indexed")

---
## Observations and Takeaways

_Fill in after running the notebook with actual results._

### Phase 1: Chunk Size

1. **Which chunk size produced the best search results overall?**
   - _TODO_

2. **How much did result sets overlap between configurations?**
   - Average Jaccard similarity: _TODO_
   - Are smaller/larger chunks retrieving fundamentally different documents?

3. **Did larger chunks help with conceptual queries?**
   - _TODO: Compare vector-excelling query results across configs_

4. **Did smaller chunks help with exact-match queries?**
   - _TODO: Compare text-excelling query results across configs_

### Phase 2: Overlap Sensitivity

5. **How sensitive are results to overlap size?**
   - _TODO_

6. **Is 0% overlap noticeably worse (lost context at boundaries)?**
   - _TODO_

7. **Does 40% overlap improve results enough to justify the larger index?**
   - _TODO_

### Recommendation

8. **Should we update the default chunk_size and chunk_overlap?**
   - _TODO: Recommended configuration based on the above analysis_